<a href="https://colab.research.google.com/github/Master-utsav/ML_2_Email_Spam_Detection/blob/main/ML_1_2_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df1 = pd.read_csv("email.csv")
df2 = pd.read_csv("emails.csv")

In [ ]:
df1 = df1.rename(columns={
    "Message": "text",
    "Category": "spam"
})

df1["spam"] = df1["spam"].map({
    "ham": 0,
    "spam": 1
})

#remove the df2 subject substr
df2["text"] = df2["text"].str.replace(r"^Subject:\s*", "", regex=True)

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=["text", "spam"])
df = df.drop_duplicates()
df['spam'] = df['spam'].astype(int)

df.to_csv("final_emails.csv", index=False)

In [ ]:
# NLP of text
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
# using tfid to create vectors
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = tfidf_vectorizer.fit_transform(df['text'])
y = df['spam']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Multinomial Naive Bayes model
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

MultinomialNB()

In [ ]:
# predictions on the test set
y_pred = mnb.predict(X_test)

In [ ]:
# model evlaution
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9682

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1740
           1       0.97      0.87      0.92       431

    accuracy                           0.97      2171
   macro avg       0.97      0.93      0.95      2171
weighted avg       0.97      0.97      0.97      2171


Confusion Matrix:
[[1729   11]
 [  58  373]]


In [ ]:
# using logistic reg
from sklearn.linear_model import LogisticRegression
model_lg = LogisticRegression()
model_lg.fit(X_train, y_train)

LogisticRegression()

In [ ]:
y_pred_lg = model_lg.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_lg):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lg))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lg))

Accuracy: 0.9562

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1740
           1       0.97      0.81      0.88       431

    accuracy                           0.96      2171
   macro avg       0.96      0.90      0.93      2171
weighted avg       0.96      0.96      0.95      2171


Confusion Matrix:
[[1729   11]
 [  84  347]]


In [ ]:
# using SVM
from sklearn.svm import SVC
model_svm = SVC()
model_svm.fit(X_train, y_train)

SVC()

In [ ]:
y_pred_svm = model_svm.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Accuracy: 0.9724

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1740
           1       0.98      0.88      0.93       431

    accuracy                           0.97      2171
   macro avg       0.98      0.94      0.95      2171
weighted avg       0.97      0.97      0.97      2171


Confusion Matrix:
[[1733    7]
 [  53  378]]


# Highest accuracy with low recall
we will use SVM model here

In [ ]:
# export the model with tf-idf
import joblib
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("tfidf", tfidf_vectorizer),
    ("model", model_svm)
])

pipeline.fit(df["text"], df["spam"])

joblib.dump(pipeline, "spam_predictor_model.joblib")

✅ Pipeline model saved successfully
